# Pilot10 meta-analysis: which dataset properties predict a quantum advantage?

This notebook asks whether pass-level meta-features (MFE complexity measures and task
statistics) explain where the quantum arm does better or worse than the classical arm.
Every step is chosen so that a reviewer cannot reject it for one of the standard reasons:

| Reviewer objection | What this notebook does |
|---|---|
| "Rows are not independent" | Iterations and the two embeddings of one dataset share splits, so **the dataset is the cluster** (G = 12). Every p-value is cluster-robust and small-G corrected. |
| "The winner is picked on the data it is scored on" | The response is the **leave-one-iteration-out (LOIO)** contrast: each side's arm is chosen on the other iterations and scored on the held-out one. |
| "Features were chosen by looking at y" | The screen (reliability, host noise, redundancy, **VIF**) never sees the response. Its whole ledger is printed. |
| "141 features, 16 passes: the model is not identified" | One pre-screened feature per model (marginal models), BH/BY FDR across the family. A joint model runs only if G >= 10 x k. |
| "Your bootstrap is anti-conservative with 12 clusters" | Primary test = **wild cluster restricted bootstrap-t** (Webb weights), checked against CV3 jackknife and an exact block permutation. **Every method is calibrated on this design** (Section 9). |
| "Null results only mean low power" | Power and minimum detectable effect on this design (Section 10). |
| "One dataset drives it" | Leave-one-dataset-out influence, synthetic exclusion, alternative metrics, matched pairs, alternative screen thresholds, budget-ratio nuisance, freeze diagnostic (Section 11). |

The group-bootstrap ridge that was proposed is implemented as posed (Section 8) and
calibrated beside the primary method (Section 9), so the reason for not making it the
primary test is a measured one, not an opinion.

**Pre-specification.** The parameters cell below is the analysis plan. The screen is
outcome-blind. The primary p-value, the FDR family and the effect scale (qadv per pass-level SD) are
fixed before any test is run. Section 0 prints a hash of the input so the numbers can be tied
to one file.

In [ ]:
# Parameters: the analysis plan. Change them here, not below.
RESULTS_CSV = "collated/ModelResults.csv"
CONFIG_DIR = "configs"
METRIC = "balanced_accuracy"            # primary; robust to the class imbalance in this corpus
ALT_METRICS = ["mcc", "auc", "f1_score"]
ALPHA = 0.05
FDR_Q = 0.05                             # BH level of the meta-regression feature family
# Per-dataset verdicts (select_winners). EPS, the TOST equivalence bound, is derived in
# Section 3 from the resolution floor; it no longer decides wins.
MARGIN = 0.0        # pre-registered superiority margin: a win needs the whole CI beyond it
VERDICT_FDR = 0.10  # BH level of the adjusted per-dataset verdicts, across all datasets
SEED = 20260929
SCREEN = dict(min_reliability=0.7, max_host_noise=0.25, max_top_share=0.8,
              log_range=100.0, redundancy=0.8, vif_max=5.0, max_features=None)
ALT_SCREENS = {"strict (VIF 2.5, rho 0.7)": dict(SCREEN, vif_max=2.5, redundancy=0.7),
               "loose (VIF 10, rho 0.9)": dict(SCREEN, vif_max=10.0, redundancy=0.9)}
N_BOOT = 9999          # wild bootstrap draws for the primary table
N_PERM = 9999          # block permutations for the primary table
N_BOOT_SENS = 1999     # for sensitivity analyses
N_PERM_RIDGE = 999
N_BOOT_RIDGE = 1999
N_NULL = 200           # null responses for calibration
B_NULL = 399           # bootstrap draws per null response
POWER_EFFECTS = (0.0, 0.025, 0.05, 0.075, 0.10, 0.15, 0.20)
N_SIM_POWER = 100
SYNTHETIC = r"^(eng|gs|hl|ql|te)_"
# Variants generated from one base share a generator, so they are one cluster
# (mr.dataset_family). Optional explicit stem -> cluster CSV. None = rules only. The draft
# "../cluster_map_draft.csv" is not frozen and its stem column is ambiguous (breast_cancer
# is two datasets), so it raises until the dedup list resolves that; see dataset_family.
CLUSTER_MAP = None
MATCHED_PAIRS = [("qsvc_opt", "svc_opt"), ("pqk_opt", "svc_opt")]

In [ ]:
import hashlib, importlib.util, logging, os, re, sys, time
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from qbiocode.utils import meta_regression as mr
from qbiocode.utils.fair_selection import model_side, select_winners
from qbiocode.utils.qc_winner_finder import resolution_floor_epsilon

logging.basicConfig(level=logging.WARNING, format="%(name)s: %(message)s", force=True)
logging.getLogger("qbiocode.utils.meta_regression").setLevel(logging.INFO)
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_colwidth", 90)
T0 = time.time()

# Chart tokens: the reference data-viz palette (slots 1-3 validate all-pairs).
INK, INK2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#8a8984", "#e6e5e1", "#fcfcfb"
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
EMB_COLOR = {"none": BLUE, "pca": ORANGE, "umap": AQUA}
mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.grid": True, "grid.color": GRID, "grid.linestyle": "-",
    "grid.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "axes.spines.left": False, "axes.axisbelow": True, "lines.linewidth": 2, "lines.markersize": 8,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "legend.frameon": False,
    "figure.dpi": 110,
})

def icc1_by(values, groups):
    """One-way ANOVA ICC(1) for unbalanced groups."""
    df = pd.DataFrame({"v": np.asarray(values, float), "g": np.asarray(groups)})
    k, n = df.groupby("g").size(), len(df)
    a = len(k)
    msb = (k * (df.groupby("g")["v"].mean() - df["v"].mean()) ** 2).sum() / (a - 1)
    msw = ((df["v"] - df.groupby("g")["v"].transform("mean")) ** 2).sum() / (n - a)
    m0 = (n - (k ** 2).sum() / n) / (a - 1)
    return float((msb - msw) / (msb + (m0 - 1) * msw))

def fmt_p(p):
    return "n/a" if not np.isfinite(p) else (f"{p:.4f}" if p >= 1e-4 else "<1e-4")

## 0. Input and provenance

In [ ]:
with open(RESULTS_CSV, "rb") as fh:
    digest = hashlib.sha256(fh.read()).hexdigest()
results = pd.read_csv(RESULTS_CSV, low_memory=False)
test_size_cfg = n_iter_cfg = None
spec = importlib.util.spec_from_file_location("analyze_pilot", "analyze_pilot.py")
analyze_pilot = importlib.util.module_from_spec(spec)
spec.loader.exec_module(analyze_pilot)
TEST_SIZE, N_ITER, CONFIGS = analyze_pilot.read_run_protocol(CONFIG_DIR)
print(f"{RESULTS_CSV}  sha256 {digest[:16]}...  {results.shape[0]} rows x {results.shape[1]} columns")
print(f"run protocol from {len(CONFIGS)} configs: test_size = {TEST_SIZE}, iter = {N_ITER}")
print(f"numpy {np.__version__}, pandas {pd.__version__}, python {sys.version.split()[0]}")

## 1. What the ModelResults CSV looks like

One row per **(Dataset, embeddings, iteration, model)**. Each row carries

* the held-out metrics of that model on that resample (`balanced_accuracy`, `mcc`, `auc`, ...),
* the tuned parameters (`Model_Parameters`, `BestParams_*`), and
* **~140 meta-features of the dataset pass** (`mfe.*`, `task.*`, a few summary statistics),
  repeated on every model row of the same instance.

So the meta-features vary between passes, and a little between model rows because each row was
computed in its own job on its own host. They do not vary with the model. The response we
want to explain is a contrast *between* models, so the natural unit is the instance
(Dataset x embeddings x iteration), and the covariates live one level up, on the pass.

In [ ]:
meta_cols = mr.meta_feature_columns(results)
other = [c for c in results.columns if c not in meta_cols]
print(f"{len(meta_cols)} numeric meta-feature columns; the other {len(other)} columns:")
print(", ".join(other))
design_tbl = (results.groupby(["Dataset", "embeddings"])
              .agg(iterations=("iteration", "nunique"), models=("model", "nunique"), rows=("model", "size"))
              .reset_index())
display(design_tbl)
counts = results.groupby(list(mr.UNIT) + ["model"]).size()
short = counts[counts != N_ITER]
print(f"{len(design_tbl)} passes over {results['Dataset'].nunique()} datasets, "
      f"{results['model'].nunique()} models, {results.groupby(list(mr.INSTANCE)).ngroups} instances.")
print("arms with a missing iteration:" if len(short) else "every arm has every iteration.")
if len(short):
    display(short.rename("iterations").reset_index())

## 2. Meta-feature quality: canonical values, reliability, host noise

Before any feature can be a covariate, three things are checked, all without the response:

* **Canonical value.** The model rows of one instance do not always agree (the kNN and graph
  measures move with the host's CPU type). The instance value is the *median* over its model
  rows, which does not depend on row order.
* **Reliability.** A pass-level covariate is the mean over its 5 resamples. ICC(1,k) = 1 - MSW/MSB
  is the reliability of that mean. A coefficient on an unreliable covariate is attenuated
  towards zero by about that factor.
* **Host noise.** The pooled SD across one instance's model rows, as a share of the between-pass
  SD. Above 0.25, a feature differs by host nearly as much as by dataset.

In [ ]:
X_inst, quality = mr.canonical_meta_features(results)
reliability = mr.feature_reliability(X_inst, quality)
X_unit = mr.unit_means(X_inst)
print(f"canonical matrix: {X_inst.shape[0]} instances x {X_inst.shape[1]} features; "
      f"pass means: {X_unit.shape[0]} passes")
q = reliability
summary = pd.Series({
    "features with any non-finite raw value": int((q["nonfinite_rows"] > 0).sum()),
    "reliability < 0.7": int((q["reliability"] < SCREEN["min_reliability"]).sum()),
    "reliability not estimable": int(q["reliability"].isna().sum()),
    "host_noise > 0.25": int((q["host_noise"] > SCREEN["max_host_noise"]).sum()),
    "median reliability": round(float(q["reliability"].median()), 3),
    "median host_noise": round(float(q["host_noise"].median()), 4),
})
display(summary.to_frame("value"))
display(q.sort_values("host_noise", ascending=False)
         [["reliability", "icc1", "host_sd", "between_sd", "host_noise", "nonfinite_rows"]].head(8))

## 3. How quantum winners are found, and how they should be found

**The current rule (`qc_winner_finder` / `select_winners`).** For each dataset, the arms
(model x embedding) of each side are compared. The corrected rule is
**leave-one-iteration-out (LOIO)**: for every held-out iteration, the best arm of each side is
chosen on the *other* iterations and scored on the held-out one. The per-dataset delta
(`classical - quantum`, so negative = quantum) gets a Nadeau-Bengio corrected t interval,
which accounts for overlapping train/test splits. The raw verdict is a win when the whole interval
clears the pre-registered superiority margin `MARGIN` (0 here), `equivalent` when it lies inside
(-epsilon, +epsilon) (TOST), and `inconclusive` otherwise. `p_value` tests |delta| <= `MARGIN`;
it is BH-adjusted across all datasets at `VERDICT_FDR`, which gives `verdict_adjusted`. The corpus
claim is made across datasets (t, Wilcoxon, sign test).

**Why not the pooled maximum?** Picking the best model and reporting its own mean is the
winner's curse. With 4 quantum and 9 classical models per embedding, the side with more arms wins on noise.
`naive_delta` below is that pooled rule, and the gap `naive_delta - delta` is the bias LOIO removes.

**The equivalence bound.** `resolution_floor_epsilon` sets epsilon to the smallest half-width the
design can certify. It pools a model's resample SD within each (dataset, embedding) pass, so the
pca/umap gap no longer inflates it; the frame relabelled `Dataset := ds|emb` gives the same value,
which the cell below checks.

In [ ]:
EPS, s_pass = resolution_floor_epsilon(results, METRIC, TEST_SIZE, ALPHA)
# resolution_floor_epsilon groups by embedding, so it equals the value on the per-pass
# relabelled frame. Before that fix it pooled pca and umap (0.0688 vs 0.0633 here).
eps_units, _ = resolution_floor_epsilon(mr._as_units(results), METRIC, TEST_SIZE, ALPHA)
assert np.isclose(EPS, eps_units), (EPS, eps_units)
print(f"epsilon (resolution floor, per pass), {METRIC}: {EPS:.4f} (s = {s_pass:.4f});  "
      f"margin {MARGIN:g}, verdict FDR {VERDICT_FDR:g}")

per_dataset_report = select_winners(results, metric=METRIC, epsilon=EPS, alpha=ALPHA,
                                    test_size=TEST_SIZE, margin=MARGIN, fdr=VERDICT_FDR)
pdr = per_dataset_report.per_dataset
cols = ["Dataset", "classical_mean", "quantum_mean", "delta", "naive_delta", "ci_lo", "ci_hi",
        "p_value", "within_equivalence", "verdict_raw", "p_adjusted", "verdict_adjusted"]
display(pdr[[c for c in cols if c in pdr]].round(4))
corp_all = per_dataset_report.corpus
corp = corp_all["across_datasets"]
print(f"verdicts (raw): {corp_all['counts_raw']};  after BH: {corp_all['counts_adjusted']}")
print(f"median per-dataset 95% half-width {corp_all['median_interval_half_width']:.4f} vs epsilon {EPS:.4f}: "
      f"a single dataset {'cannot' if corp_all['median_interval_half_width'] > EPS else 'can'} usually certify equivalence")
print(f"quantum winners after FDR: {per_dataset_report.quantum_datasets or 'none'};  "
      f"classical winners: {per_dataset_report.classical_datasets or 'none'}")
print(f"corpus: mean delta {corp['mean_delta']:+.4f} (classical - quantum), "
      f"95% CI [{corp['ci'][0]:+.4f}, {corp['ci'][1]:+.4f}], t p = {fmt_p(corp['t']['p_two_sided'])}, "
      f"Wilcoxon p = {fmt_p(corp['wilcoxon']['p_two_sided'])}, sign test { {k: (round(v, 4) if isinstance(v, float) else v) for k, v in corp['sign'].items()} }, "
      f"direction: {corp['direction']}")
bias = (pdr["naive_delta"] - pdr["delta"])
print(f"winner's-curse gap (naive - LOIO): mean {bias.mean():+.4f}, range [{bias.min():+.4f}, {bias.max():+.4f}]")

The **per-pass** report runs the same rule once per (Dataset, embeddings). It supplies the response
for the meta-regression, because a pass-level covariate can only explain a pass-level contrast.
Its per-pass verdicts are descriptive. A dataset's pca and umap passes share splits, so the corpus
claim above stays at the dataset level.

Arms that never beat chance are listed as well. A quantum arm stuck at 0.5 makes the LOIO pick
fall to the other quantum arms, which is legitimate. A reader should still know it happened.

In [ ]:
unit_rep = mr.unit_report(results, metric=METRIC, epsilon=EPS, alpha=ALPHA,
                          test_size=TEST_SIZE, margin=MARGIN, fdr=VERDICT_FDR)
upd = unit_rep.per_dataset
y_units_all = upd[["Dataset", "embeddings"]].drop_duplicates()
display(upd[[c for c in cols[:1] + ["embeddings"] + cols[1:] if c in upd]].round(4))
deg = mr.degenerate_arms(results, METRIC)
deg["side"] = deg["model"].map(model_side)
at_chance = deg[deg["at_chance"]]
print(f"{len(at_chance)} arm-passes never beat chance ({METRIC} <= 0.5 on every iteration):")
with pd.option_context("display.max_colwidth", None):
    display(at_chance.groupby(["side", "model"])
            .apply(lambda g: pd.Series({
                "n_passes": f"{len(g)} of {len(y_units_all)}",
                "passes": ", ".join(f"{d.replace('.csv', '')}/{e}" for d, e in zip(g.Dataset, g.embeddings))}),
                include_groups=False).reset_index())

## 4. The response: per-instance quantum advantage

`qadv = quantum_score - classical_score` on each held-out instance, where each score belongs to
the arm its side picked on the other iterations. **Positive = quantum ahead**, so a positive
coefficient reads "this property favours quantum". The sign is the reverse of `delta` above.

In [ ]:
contrast = mr.loio_contrast(unit_rep)
y_unit = contrast.groupby(list(mr.UNIT))["qadv"].mean()
y_vd = mr.variance_decomposition(contrast.set_index(list(mr.INSTANCE))[["qadv"]])
y_rel = mr.feature_reliability(contrast.set_index(list(mr.INSTANCE))[["qadv"]])
print(f"qadv over {len(contrast)} instances: mean {contrast.qadv.mean():+.4f}, SD {contrast.qadv.std():.4f}; "
      f"pass means SD {y_unit.std():.4f}")
print(f"share of qadv variance between datasets {y_vd.loc['qadv', 'between_datasets']:.2f}, "
      f"between passes {y_vd.loc['qadv', 'between_passes']:.2f}; ICC(1) {y_rel.loc['qadv', 'icc1']:.2f}, "
      f"reliability of a pass mean {y_rel.loc['qadv', 'reliability']:.2f}")

order = y_unit.sort_values().index
fig, ax = plt.subplots(figsize=(7.5, 0.34 * len(order) + 1.2))
for i, (d, e) in enumerate(order):
    pts = contrast[(contrast.Dataset == d) & (contrast.embeddings == e)]["qadv"]
    ax.scatter(pts, np.full(len(pts), i), s=22, color=EMB_COLOR[e], alpha=0.45, linewidths=0)
    ax.scatter([pts.mean()], [i], s=64, color=EMB_COLOR[e], edgecolors=SURFACE, linewidths=2, zorder=3)
ax.axvline(0, color=INK2, linewidth=1)
ax.set_yticks(range(len(order)), [f"{d.replace('.csv', '')} / {e}" for d, e in order])
ax.grid(axis="y", visible=False)
ax.set_xlabel(f"qadv = quantum - classical {METRIC} (LOIO)")
ax.set_title("Quantum advantage per pass: 5 held-out iterations and their mean", loc="left")
for e, c in EMB_COLOR.items():
    ax.scatter([], [], s=64, color=c, label=e)
ax.legend(title="embedding", loc="lower right")
fig.tight_layout(); plt.show()

## 5. The blocker, quantified

The proposal was a ridge regression on dataset x embedding x iteration rows, with a group
bootstrap for p-values and FDR. The blocker is not the software. It is the information in the design:

In [ ]:
G = results["Dataset"].nunique()
U = len(X_unit)
N = len(contrast)
# Standardise first: raw scales span ~1e-3 to ~1e18, which would swamp the rank tolerance.
usable = X_unit.loc[:, np.isfinite(X_unit).all() & (X_unit.std() > 0)]
rank = np.linalg.matrix_rank(((usable - usable.mean()) / usable.std()).to_numpy())
feat_vd = mr.variance_decomposition(X_inst)
between = feat_vd["between_passes"].dropna()
icc_y = icc1_by(contrast["qadv"], contrast["Dataset"])
m_bar = N / G
n_eff = N / (1 + (m_bar - 1) * max(icc_y, 0.0))
blocker = pd.DataFrame([
    ("independent clusters (datasets), G", G, "the only replication that informs a dataset-level slope"),
    ("distinct covariate rows (passes), U", U, "features are constant within a pass"),
    ("regression rows (instances), N", N, "iterations replicate y, not x"),
    ("candidate meta-features, p", X_unit.shape[1], ""),
    (f"rank of the standardised pass x feature matrix ({usable.shape[1]} usable features)", rank,
     "<= U - 1: a joint fit of more features than this is not identified"),
    ("median share of feature variance between passes", round(float(between.median()), 3),
     "i.e. iterations add essentially no x information"),
    ("ICC(1) of qadv, clustering by dataset", round(icc_y, 3), "rows of one dataset are strongly alike"),
    ("design-effect effective n for mean qadv", round(n_eff, 1),
     "N / (1 + (m-1) ICC), m = rows per dataset; a pass-level slope has at most G"),
], columns=["quantity", "value", "why it matters"])
with pd.option_context("display.max_colwidth", None):
    display(blocker)
pairs_distinct = G * (1 - (1 - 1 / G) ** G)
print(f"a pairs cluster bootstrap draws on average {pairs_distinct:.1f} distinct datasets of {G} per replicate")

What follows from these numbers:

1. **Effective n is G = 12, not N = 80.** The covariates are pass-level, so the 5 iterations only
   average the response noise. A row-level ridge with iid (or row-bootstrap) standard errors counts
   each dataset 5-10 times. Section 9 measures what that does to iid OLS on a true null on this
   design.
2. **p >> U.** With 141 features and 16 passes the matrix has rank 15, so every VIF is infinite
   and a joint coefficient is not identified. Ridge returns numbers, but they are chosen by the
   penalty, not the data. Ridge also has no valid coefficient p-values, because the estimator is
   biased by construction and the percentile interval of a biased estimator does not have its
   nominal coverage.
3. **A pairs cluster bootstrap with 12 clusters** draws only about 7.8 distinct datasets per
   replicate (G(1 - (1 - 1/G)^G) of them, printed above), so some replicates cannot identify the embedding
   dummies. Its documented failure at G ~ 10 is over-rejection (Cameron, Gelbach & Miller 2008).
   Whether that happens *here* is measured, not assumed (Section 9). The primary test is the
   **wild cluster restricted bootstrap-t**, which that literature recommends for few clusters. It keeps all G
   clusters in every replicate, imposes the null, and bootstraps a pivotal t, so its validity does not
   depend on how the design happens to fall.
4. **Feature filtering must not look at y.** VIF filtering is the right instinct. Here it runs
   last in an outcome-blind screen, so the tested family is fixed before any p-value is computed
   and the FDR adjustment is honest.

## 6. Outcome-blind screen, ending in VIF admission

Stages: non-finite -> near-constant -> unreliable (ICC(1,k) < 0.7) -> host-dependent -> log10 for
wide positive features -> z-score -> redundancy clusters (average linkage on 1 - |Spearman rho|,
cut at |rho| = 0.8, keep the medoid) -> **forward VIF admission (VIF <= 5 with the embedding
dummies in the design)** -> degree-of-freedom cap. The response is not an argument of the screen.

In [ ]:
nuisance = mr.embedding_indicators(X_unit.index)
screen = mr.screen_features(X_unit, reliability, nuisance, **SCREEN)
funnel = screen.funnel()
display(funnel.T)
print(f"selected ({len(screen.selected)}): {screen.selected}")
print(f"final VIFs: " + ", ".join(f"{k} {v:.2f}" for k, v in screen.vif.items()))

fig, ax = plt.subplots(figsize=(7.0, 3.0))
ax.barh(range(len(funnel)), funnel["n_features"], color=BLUE, height=0.6)
for i, v in enumerate(funnel["n_features"]):
    ax.text(v + 1, i, str(v), va="center", color=INK2)
ax.set_yticks(range(len(funnel)), funnel["stage"])
ax.invert_yaxis(); ax.grid(axis="y", visible=False)
ax.set_xlabel("features per stage (the last row is the admitted set)")
ax.set_title(f"Screen funnel: {X_unit.shape[1]} candidates -> {len(screen.selected)} admitted", loc="left")
fig.tight_layout(); plt.show()

In [ ]:
led = screen.ledger
display(led[led["stage"].isin(["selected", "vif", "cap"])]
        .sort_values(["stage", "cluster"])[["stage", "detail", "reliability", "transform", "cluster", "cluster_size", "vif"]])
display(screen.clusters[screen.clusters["size"] > 1].head(15))

Each admitted feature stands for its whole redundancy cluster. Read a hit on `mfe.sd.mean` as
"the attribute-scale cluster", not as that single statistic.

## 7. Primary analysis: one marginal model per admitted feature

For each admitted feature f: `qadv ~ 1 + emb[pca] + emb[umap] + z_f` on all instance rows,
clustered by dataset. Columns:

* `beta`: change in qadv per pass-level SD of the feature.
* `ci_lo/ci_hi`: CV3 (cluster jackknife) t interval on G - 1 df.
* `p_wcr`: **primary**, wild cluster restricted bootstrap-t with Webb weights.
* `p_perm`: Monte Carlo block permutation of whole datasets within same-shape strata
  (the strata and their number of orderings are printed first).
* `p_cr1`, `p_cv3`: analytic cluster-robust t on G - 1 df.
* `p_ols_iid`: rows treated as independent. Shown only to display the error.
* `q_bh`, `q_by`: BH and BY adjustment of `p_wcr` across the family.

In [ ]:
design = mr.build_design(contrast, screen.features, nuisance)
print(f"design: N = {design.N} rows, G = {design.G} clusters, nuisance = {list(design.nuisance.columns)}")
display(mr.permutation_strata(design))
primary = mr.marginal_tests(design, n_boot=N_BOOT, n_perm=N_PERM, alpha=ALPHA, seed=SEED)
display(primary.round(4))
n_hits = int((primary["q_bh"] <= FDR_Q).sum())
print(f"discoveries at BH q <= {FDR_Q}: {n_hits};  smallest p_wcr {primary.p_wcr.min():.4f} "
      f"({primary.loc[primary.p_wcr.idxmin(), 'feature']})")
joint = mr.joint_tests(design, n_boot=N_BOOT, seed=SEED)
print("joint model:", "skipped, since G = %d < 10 x %d features" % (design.G, design.features.shape[1])
      if joint is None else "")
if joint is not None:
    display(joint.round(4))

In [ ]:
tab = primary.sort_values("beta").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(7.5, 0.42 * len(tab) + 1.3))
ax.hlines(range(len(tab)), tab.ci_lo, tab.ci_hi, color=BLUE, linewidth=2)
ax.scatter(tab.beta, range(len(tab)), s=64, color=BLUE, edgecolors=SURFACE, linewidths=2, zorder=3)
ax.axvline(0, color=INK2, linewidth=1)
for i, r in tab.iterrows():
    ax.text(ax.get_xlim()[1], i, f"  p = {r.p_wcr:.2f}", va="center", color=INK2, fontsize=9)
ax.set_yticks(range(len(tab)), tab.feature)
ax.grid(axis="y", visible=False)
ax.set_xlabel("change in qadv per pass-level SD (95% CV3 interval, G - 1 df)")
ax.set_title("Marginal meta-regression coefficients; p = wild cluster bootstrap", loc="left")
fig.tight_layout(); plt.show()

## 8. Ridge, used where it is valid, and the proposal as posed

**Ridge as a predictor.** Do the admitted features, *jointly*, predict qadv on a dataset the model
never saw? Nested leave-one-dataset-out CV: the penalty is chosen inside each training fold, and
the embedding dummies are unpenalised. The statistic is Delta R^2 over the nuisance-only model. Its
p-value comes from refitting the *whole* pipeline, penalty search included, on block-permuted
responses. That single omnibus test replaces the per-coefficient p-values ridge cannot supply.

**The proposal as posed.** Joint ridge on all rows, datasets resampled with replacement,
percentile p-values, BH. It is reported for comparison, and Section 9 measures its type-I error.

In [ ]:
omni = mr.ridge_omnibus(design, n_perm=N_PERM_RIDGE, seed=SEED)
print(f"LODO R^2: features + nuisance {omni.r2_full:+.3f}, nuisance only {omni.r2_nuisance:+.3f}, "
      f"Delta R^2 {omni.delta_r2:+.3f}, permutation p = {omni.p_value:.3f} ({omni.n_perm} block permutations)")
print("penalty chosen per outer fold (x n_train):", np.array2string(omni.chosen_grid, precision=3))
display(omni.per_cluster.round(5).sort_values("gain"))
fig, ax = plt.subplots(figsize=(7.0, 2.8))
ax.hist(omni.null_delta_r2, bins=40, color=BLUE, alpha=0.85, edgecolor=SURFACE, linewidth=1)
ax.axvline(omni.delta_r2, color=INK, linewidth=2)
ax.text(omni.delta_r2, ax.get_ylim()[1] * 0.95, f"observed {omni.delta_r2:+.3f}  ", color=INK, ha="right", va="top",
        bbox=dict(facecolor=SURFACE, edgecolor="none", pad=1.5))
ax.grid(axis="x", visible=False)
ax.set_xlabel("Delta R^2 (leave-one-dataset-out) under block permutation")
ax.set_title("Ridge omnibus: observed gain against its permutation null", loc="left")
fig.tight_layout(); plt.show()

proposal = mr.ridge_pairs_bootstrap(design, n_boot=N_BOOT_RIDGE, seed=SEED)
display(proposal.round(4))
shrink = float(np.median(np.abs(proposal["beta_ridge"].to_numpy())) / np.median(np.abs(primary["beta"].to_numpy())))
print(f"median |ridge coefficient| is {shrink:.3f} x the median |marginal OLS coefficient|: the proposal tests a "
      f"penalty-shrunk joint coefficient, whose size depends on the chosen penalty, not on the data alone")
print(f"proposal: penalty grid value {proposal.attrs['penalty_grid_value']:.3g} (grid max {mr.RIDGE_GRID.max():g}); "
      f"singular replicates {proposal.attrs['frac_singular']:.1%}; "
      f"distinct datasets per replicate {proposal.attrs['distinct_clusters']:.2f} of {design.G}")

## 9. Calibration on this design: which p-values can be believed?

The responses are block-permuted, so every dataset's full response block, with its real
within-dataset dependence, moves onto another dataset of the same shape. The features are the real
ones, and no feature carries signal. Every method is then run as it would be on real data. A valid
method rejects at 5%, within the Monte Carlo band shown. `fdr_bh` is the chance that BH across the
family makes at least one (false) discovery.

In [ ]:
calib = mr.null_calibration(design, n_null=N_NULL, n_boot=B_NULL, alpha=ALPHA, q=FDR_Q, seed=SEED)
cal = calib.rejection()
display(cal.round(3))
fig, ax = plt.subplots(figsize=(9.0, 3.2))
band = 2 * cal["mc_se"].iloc[0]
ax.axvspan(max(ALPHA - band, 0.0), ALPHA + band, color=GRID, alpha=0.8, linewidth=0)
ax.axvline(ALPHA, color=INK2, linewidth=1)
ax.hlines(range(len(cal)), cal.type1_min, cal.type1_max, color=BLUE, linewidth=2, alpha=0.45)
ax.scatter(cal.type1, range(len(cal)), s=64, color=BLUE, edgecolors=SURFACE, linewidths=2, zorder=3)
for i, (v, hi) in enumerate(zip(cal.type1, cal.type1_max)):
    ax.text(hi + 0.01, i, f"{v:.3f}", va="center", color=INK2, fontsize=9)
ax.set_yticks(range(len(cal)), cal.method)
ax.set_xlim(0, max(0.2, cal.type1_max.max() + 0.08))
ax.invert_yaxis(); ax.grid(axis="y", visible=False)
ax.set_xlabel(f"rejection rate of a true null at alpha = {ALPHA}")
ax.set_title(f"Type-I error of each method on {N_NULL} null responses", loc="left", pad=18)
ax.text(0, 1.02, f"dot = mean over features, line = range over features, band = {ALPHA} +- 2 Monte Carlo SE",
        transform=ax.transAxes, color=INK2, fontsize=9)
fig.tight_layout(); plt.show()

## 10. Power and minimum detectable effect

A known effect of `effect x z_f` is added to block-permuted responses. The full primary
analysis, all features plus BH, is then rerun. Effects are in qadv units per pass-level SD. For scale,
the SD of the pass-mean qadv is printed above (Section 4).

In [ ]:
power = mr.power_curve(design, effects=POWER_EFFECTS, n_sim=N_SIM_POWER, n_boot=B_NULL,
                       alpha=ALPHA, q=FDR_Q, seed=SEED)
pw = power.groupby("effect")[["power", "power_bh"]].agg(["mean", "min", "max"])
display(pw.round(3))
fig, ax = plt.subplots(figsize=(7.0, 3.2))
for col, color, name in (("power", BLUE, "unadjusted"), ("power_bh", ORANGE, "after BH")):
    ax.fill_between(pw.index, pw[(col, "min")], pw[(col, "max")], color=color, alpha=0.12, linewidth=0)
    ax.plot(pw.index, pw[(col, "mean")], color=color, marker="o", label=name)
    ax.text(pw.index[-1], pw[(col, "mean")].iloc[-1], f"  {name}", color=INK2, va="center")
ax.axhline(0.8, color=INK2, linewidth=1)
ax.set_ylim(0, 1); ax.set_xlim(0, max(POWER_EFFECTS) * 1.25)
ax.set_xlabel("true effect: qadv per pass-level SD of one feature")
ax.set_ylabel("power")
ax.set_title("Power of the primary test (mean over features, band = range)", loc="left")
ax.legend(loc="upper left")
fig.tight_layout(); plt.show()
mde = pw[(("power_bh", "mean"))]
reach = mde[mde >= 0.8]
MDE80 = float(reach.index.min()) if len(reach) else np.nan
print("effect needed for 80% power after BH:",
      f"{MDE80:.3f}" if np.isfinite(MDE80) else f"> {max(POWER_EFFECTS)} (not reached on the grid)")

## 11. Robustness

Every sensitivity analysis reuses the features admitted by the primary screen, on their
primary scale, except 11e, which reruns the screen. Each table reports the primary p-value
and its BH q within that analysis.

In [ ]:
def rerun(d, label, feats=None, n_boot=N_BOOT_SENS):
    t = mr.marginal_tests(d, feats, n_boot=n_boot, n_perm=0, seed=SEED)
    t.insert(0, "analysis", label)
    return t[["analysis", "feature", "beta", "ci_lo", "ci_hi", "p_cv3", "p_wcr", "q_bh"]]

sens = []
# 11a. leave-one-dataset-out influence on each coefficient
lodo = mr.lodo_influence(design)
display(Markdown("**11a. Leave-one-dataset-out influence**"))
display(lodo.round(4))
print(f"features whose coefficient changes sign when one dataset is dropped: "
      f"{int((lodo.sign_flips > 0).sum())} of {len(lodo)}; most influential dataset overall: "
      f"{lodo.most_influential.value_counts().idxmax()} ({lodo.most_influential.value_counts().max()} features)")

In [ ]:
# 11b. synthetic datasets out; 11c. generator families as clusters
is_syn = design.rows["Dataset"].str.contains(SYNTHETIC).to_numpy()
if is_syn.any() and (~is_syn).sum():
    d_real = design.subset(~is_syn)
    print(f"11b: dropping {sorted(design.rows.Dataset[is_syn].unique())}: G = {d_real.G}")
    sens.append(rerun(d_real, f"11b real only (G={d_real.G})"))
d_fam = design.with_clusters(lambda d: mr.dataset_family(d, CLUSTER_MAP))
print(f"11c: generator families as clusters: G = {d_fam.G} (pilot has one variant per family, "
      f"so this matters only for the full corpus)")
if d_fam.G != design.G:
    sens.append(rerun(d_fam, f"11c families (G={d_fam.G})"))
# 11d. without the embedding nuisance
d_plain = mr.build_design(contrast, screen.features)
sens.append(rerun(d_plain, "11d no emb dummies"))

In [ ]:
# 11e. alternative screen thresholds (screen rerun; outcome-blind as before)
alt_rows = []
for name, rules in ALT_SCREENS.items():
    s = mr.screen_features(X_unit, reliability, nuisance, **rules)
    d = mr.build_design(contrast, s.features, nuisance)
    t = rerun(d, f"11e {name.split()[0]} screen")
    sens.append(t)
    alt_rows.append(dict(screen=name, n_selected=len(s.selected),
                         overlap_with_primary=len(set(s.selected) & set(screen.selected)),
                         min_p_wcr=t.p_wcr.min(), min_q_bh=t.q_bh.min()))
display(pd.DataFrame(alt_rows).round(4))

In [ ]:
# 11f. alternative metrics; 11g. pre-specified matched pairs (no selection step)
alt_resp = {}
for m in ALT_METRICS:
    if m not in results:
        continue
    rep = mr.unit_report(results, metric=m, epsilon=EPS, alpha=ALPHA, test_size=TEST_SIZE,
                         margin=MARGIN, fdr=VERDICT_FDR)
    alt_resp[m] = mr.loio_contrast(rep)
    sens.append(rerun(mr.build_design(alt_resp[m], screen.features, nuisance), f"11f {m}"))
for qm, cm in MATCHED_PAIRS:
    if not {qm, cm} <= set(results["model"]):
        continue
    mc = mr.matched_contrast(results, qm, cm, METRIC)
    sens.append(rerun(mr.build_design(mc, screen.features, nuisance), f"11g {qm.split('_')[0]}-{cm.split('_')[0]}"))

In [ ]:
# 11h. the tuning-budget confound as a nuisance: classical got 50 trials, quantum 6-32
budget = analyze_pilot.tuning_budget_report(CONFIGS)
cfg_ds = {os.path.basename(p).replace(".yaml", ""): c["file_dataset"][0] for p, c in CONFIGS}
budget["Dataset"] = budget["Config"].map(cfg_ds)
display(budget[["Dataset", "classical_trials", "quantum_trials", "ratio", "quantum_searches_once"]])
# The report rounds the ratio for display; the nuisance uses the exact trial ratio.
ratio = (budget["classical_trials"] / budget["quantum_trials"]).astype(float)
ratio.index = budget["Dataset"]
log_ratio = pd.DataFrame({"log2_budget_ratio": np.log2(ratio.reindex(X_unit.index.get_level_values(0)).to_numpy())},
                         index=X_unit.index)
if np.isfinite(log_ratio.to_numpy()).all() and log_ratio.nunique().iloc[0] > 1:
    d_budget = design.with_nuisance(log_ratio)
    sens.append(rerun(d_budget, "11h +budget"))
    b = mr.marginal_tests(mr.build_design(contrast, log_ratio, nuisance), n_boot=N_BOOT_SENS, n_perm=0, seed=SEED)
    print(f"budget ratio itself as the covariate: beta {b.beta[0]:+.4f} per log2 unit, "
          f"CV3 95% CI [{b.ci_lo[0]:+.4f}, {b.ci_hi[0]:+.4f}], p_wcr {b.p_wcr[0]:.3f}")
    corr = pd.Series({f: np.corrcoef(screen.features[f], log_ratio.iloc[:, 0])[0, 1] for f in screen.selected})
    print("correlation of each admitted feature with the log budget ratio (pass level):")
    print(corr.round(2).to_string())

**11i. Freeze diagnostic.** Every pilot config freezes the quantum parameters: the quantum search ran
once, on the first resample, and later resamples refit that configuration. The classical arm
re-searched on every resample. Two biases follow, with opposite signs:

* *handicap*: the frozen configuration was tuned for resample 1 only, so quantum does relatively
  better on resample 1;
* *leakage*: resample 1's training and validation rows overlap later resamples' test rows, so the frozen
  configuration has "seen" part of those test sets, and quantum does relatively better on resamples 2-5.

The within-pass contrast, resample 1 minus the mean of 2-5, with pass fixed effects absorbed and
datasets as clusters, says which bias dominates.

In [ ]:
first = (design.rows["iteration"] == design.rows["iteration"].min()).to_numpy().astype(float)
freeze = mr.within_unit_test(design, first, n_boot=N_BOOT, seed=SEED)
print(f"qadv(first resample) - qadv(later resamples), within pass: {freeze['beta']:+.4f} "
      f"(CR1 SE {freeze['se_cr1']:.4f}), p_wcr = {freeze['p_wcr']:.4f}, p_cr1 = {freeze['p_cr1']:.4f}")
FREEZE_READ = ("no detectable freeze effect" if freeze["p_wcr"] > ALPHA else
               ("handicap dominates: quantum relatively better where it was tuned" if freeze["beta"] > 0
                else "leakage dominates: quantum relatively better on resamples its search overlapped"))
print("reading:", FREEZE_READ)

In [ ]:
sens_tbl = pd.concat([primary.assign(analysis="primary")[["analysis", "feature", "beta", "ci_lo", "ci_hi", "p_cv3", "p_wcr", "q_bh"]]]
                     + sens, ignore_index=True)
on_primary = sens_tbl[sens_tbl["feature"].isin(screen.selected)]
wide_p = on_primary.pivot_table(index="feature", columns="analysis", values="p_wcr", sort=False)
display(Markdown("**Wild cluster bootstrap p of each primary feature across analyses** "
                 "(blank: the alternative screen did not admit it)"))
display(wide_p.round(3).T)
wide_b = on_primary.pivot_table(index="feature", columns="analysis", values="beta", sort=False)
agree = np.sign(wide_b).eq(np.sign(wide_b["primary"]), axis=0).astype(float).where(wide_b.notna())
display(Markdown("**Share of analyses (that include the feature) in which its coefficient keeps the primary sign**"))
display(pd.DataFrame({"same_sign_share": agree.mean(axis=1).round(2),
                      "analyses": wide_b.notna().sum(axis=1)}))
ANY_SENS_HIT = bool((sens_tbl["q_bh"] <= FDR_Q).any())
print(f"any BH discovery in any analysis: {ANY_SENS_HIT}; smallest q across all analyses {sens_tbl.q_bh.min():.3f}")

## 12. Conclusions (generated from the numbers above)

In [ ]:
cal_i = cal.set_index("key")
lines = [
    f"* **Bounds.** Equivalence bound epsilon = {EPS:.4f} (per-pass resolution floor); superiority "
    f"margin {MARGIN:g}; verdict BH level {VERDICT_FDR:g}. Median per-dataset half-width "
    f"{corp_all['median_interval_half_width']:.4f}, so single datasets mostly cannot certify equivalence; "
    f"the corpus test is the claim.",
    f"* **Winners.** At margin {MARGIN:g}, quantum wins after FDR: "
    f"{', '.join(per_dataset_report.quantum_datasets) or 'none'}; classical wins: "
    f"{', '.join(per_dataset_report.classical_datasets) or 'none'}. Corpus mean delta (classical - quantum) "
    f"{corp['mean_delta']:+.4f}, 95% CI [{corp['ci'][0]:+.4f}, {corp['ci'][1]:+.4f}]; direction: {corp['direction']}.",
    f"* **Design.** G = {design.G} independent datasets, {U} passes, {N} instances, {X_unit.shape[1]} candidate "
    f"features, standardised pass x feature rank {rank}. The screen admitted {len(screen.selected)} features, all with VIF <= "
    f"{SCREEN['vif_max']:g}.",
    f"* **Primary result.** {n_hits} of {len(primary)} features reach BH q <= {FDR_Q}. Smallest wild-bootstrap "
    f"p = {primary.p_wcr.min():.3f}; smallest q = {primary.q_bh.min():.3f}. The ridge omnibus Delta R^2 = "
    f"{omni.delta_r2:+.3f} (p = {omni.p_value:.3f}): the features jointly do not predict qadv on held-out datasets.",
    f"* **Calibration.** Type-I error at 5%: iid OLS {cal_i.loc['ols_iid', 'type1']:.3f}, CR1 {cal_i.loc['cr1', 'type1']:.3f}, "
    f"pairs bootstrap OLS {cal_i.loc['pairs_ols', 'type1']:.3f}, proposed pairs ridge {cal_i.loc['pairs_ridge', 'type1']:.3f}, "
    f"CV3 {cal_i.loc['cv3', 'type1']:.3f}, wild cluster bootstrap {cal_i.loc['wcr', 'type1']:.3f} "
    f"(MC SE {cal_i['mc_se'].iloc[0]:.3f}). "
    + ("The proposed pairs-bootstrap ridge holds its size on this design; its limitation is the estimand "
       f"(coefficients about {shrink:.2g} x the marginal ones, set by the penalty), not the level."
       if abs(cal_i.loc['pairs_ridge', 'type1'] - ALPHA) <= 2 * cal_i['mc_se'].iloc[0] else
       f"The proposed pairs-bootstrap ridge is off its nominal level on this design ({cal_i.loc['pairs_ridge', 'type1']:.3f})."),
    f"* **Power.** After BH, 80% power needs an effect of "
    + (f"{MDE80:.3f}" if np.isfinite(MDE80) else f"more than {max(POWER_EFFECTS)}")
    + f" qadv per SD; the observed pass-mean qadv SD is {y_unit.std():.3f}. A null here is not evidence of no effect.",
    f"* **Robustness.** Sign changes under leave-one-dataset-out: {int((lodo.sign_flips > 0).sum())} of {len(lodo)} "
    f"features. Any BH discovery in any sensitivity analysis: {ANY_SENS_HIT}. Freeze diagnostic: {FREEZE_READ} "
    f"(beta {freeze['beta']:+.4f}, p = {freeze['p_wcr']:.3f}).",
]
display(Markdown("\n".join(lines)))
print(f"notebook ran in {time.time() - T0:.0f} s")

### What the full run needs for this analysis to have a chance

* **More independent datasets, not more iterations.** Power is set by G. Iterations only shrink the
  within-dataset noise, which is already small next to the between-dataset spread. Synthetic
  variants from one generator count as one cluster (11c), so adding tau values does not add G.
* **Equal tuning budgets, or the budget ratio recorded per dataset and entered as a nuisance
  (11h).** In the pilot it is aliased with dataset size, which is itself a candidate meta-feature.
* **Per-resample quantum search, or a documented freeze.** 11i estimates the net freeze bias. It
  cannot separate the handicap from the leakage.
* **Freeze the cluster map.** `experiments/cluster_map_draft.csv` is a draft pending the frozen dedup
  list; set `CLUSTER_MAP` to the frozen file so G counts independent datasets.
* **Seeded quantum and NB tuning** (the pilot's was unseeded under Hydra) and host pinning for the
  host-sensitive arms. Otherwise part of the response is host noise.
* **Freeze this notebook's parameters cell before the full run lands**. The screen thresholds, the
  primary p-value and the FDR family then remain a pre-registered plan rather than a choice.